In [1]:
from pathlib import Path
import json
import subprocess
import shlex
import pandas as pd
from PIL import Image, ImageDraw, ImageFont

# =============================================================================
# Path setup
# =============================================================================

# Local:
# REPO_ROOT = Path("../..").resolve()

# UBELIX:
REPO_ROOT = Path("..").resolve() / "master-thesis"

HAM_ROOT = REPO_ROOT / "data" / "HAM10000"
MEL_NV_ROOT = HAM_ROOT / "mel_nv"

IMG_DIR = HAM_ROOT
MASK_ROOT = HAM_ROOT

# =============================================================================
# Input CSV
# =============================================================================

# Use your curated CSV from notebook 05b.
# This should contain image_id, image_rel_path, gt_label, mask_rel_path.
CURATION_CSV = (
    REPO_ROOT
    / "outputs"
    / "mel_nv"
    / "feature_space_difficult_cases_last_block"
    / "clinician_curation_candidates"
    / "clinician_curation_candidates_combined_cls_gap.csv"
)

# Optional: restrict to specific groups for the first review.
CURATION_GROUPS_TO_KEEP = [
    "nearest_mel_centroid",
    "nearest_nv_centroid",
    "fp_mel_top10",
    "fn_mel_all",
]

# If you only want selected model rows from the combined curation file.
# Set to None to keep all.
MODEL_SHORT_TO_KEEP = None
# Example:
# MODEL_SHORT_TO_KEEP = "gap_ha025"

# Remove duplicate image IDs across CLS/GAP curation rows.
DROP_DUPLICATE_IMAGES = True

# Optional quick debug.
MAX_IMAGES = None
# MAX_IMAGES = 5

# =============================================================================
# CAM settings
# =============================================================================

TARGET_BLOCK_INDICES = [-1, -4, -10]

PANEL_ITEMS = "rgb_gt_mask,gradcam_a,map_diff,finercam"
PANEL_SUFFIX = PANEL_ITEMS.replace(",", "_")

ALPHA = 0.8

GT_COL = "gt_label"
CLASS_ARGS = ["--class_names", "MEL,NV"]

DIRECTION_CONFIGS = [
    {
        "direction_name": "gt_direction",
        "compare_mode": "gt_pair",
        "display_name": "GT direction",
    },
    {
        "direction_name": "inverse_direction",
        "compare_mode": "inverse_gt_pair",
        "display_name": "Inverse direction",
    },
]

CLS_CKPT = REPO_ROOT / "external" / "checkpoints4" / "checkpoint-best-cls.pth"
GAP_CKPT = REPO_ROOT / "external" / "checkpoints4" / "checkpoint-best-gap.pth"

SCENARIOS = [
    {
        "name": "CLS HA 0.5",
        "short_name": "cls_ha05",
        "checkpoint": CLS_CKPT,
        "checkpoint_model_type": "panderm",
        "pooling": "cls",
    },
    {
        "name": "GAP HA 0.25",
        "short_name": "gap_ha025",
        "checkpoint": GAP_CKPT,
        "checkpoint_model_type": "panderm",
        "pooling": "mean",
    },
]

# =============================================================================
# Runtime
# =============================================================================

DRY_RUN = False
RUN_CAM_GENERATION = True
BUILD_PDFS = True

OUT_ROOT = REPO_ROOT / "outputs" / "mel_nv" / "bidirectional_cam_review"
OUT_ROOT.mkdir(parents=True, exist_ok=True)

print("REPO_ROOT:", REPO_ROOT)
print("CURATION_CSV exists:", CURATION_CSV.exists(), CURATION_CSV)
print("OUT_ROOT:", OUT_ROOT)
print("Blocks:", TARGET_BLOCK_INDICES)

for s in SCENARIOS:
    print("\n", s["name"])
    print("  checkpoint exists:", Path(s["checkpoint"]).exists(), s["checkpoint"])
    print("  pooling:", s["pooling"])

REPO_ROOT: /storage/homefs/cn21m021/projects/master-thesis
CURATION_CSV exists: True /storage/homefs/cn21m021/projects/master-thesis/outputs/mel_nv/feature_space_difficult_cases_last_block/clinician_curation_candidates/clinician_curation_candidates_combined_cls_gap.csv
OUT_ROOT: /storage/homefs/cn21m021/projects/master-thesis/outputs/mel_nv/bidirectional_cam_review
Blocks: [-1, -4, -10]

 CLS HA 0.5
  checkpoint exists: True /storage/homefs/cn21m021/projects/master-thesis/external/checkpoints4/checkpoint-best-cls.pth
  pooling: cls

 GAP HA 0.25
  checkpoint exists: True /storage/homefs/cn21m021/projects/master-thesis/external/checkpoints4/checkpoint-best-gap.pth
  pooling: mean


In [2]:
def safe_name(name: str) -> str:
    out = str(name).lower()
    for ch in [" ", "/", "\\", ":", ";", ",", "(", ")", "[", "]", "{", "}", "+", "."]:
        out = out.replace(ch, "_")
    while "__" in out:
        out = out.replace("__", "_")
    return out.strip("_")


def make_active_csv():
    if not CURATION_CSV.exists():
        raise FileNotFoundError(CURATION_CSV)

    df = pd.read_csv(CURATION_CSV)

    required = {"image_id", "image_rel_path", "gt_label", "mask_rel_path"}
    missing = required - set(df.columns)
    if missing:
        raise ValueError(f"Missing required columns in curation CSV: {missing}")

    if "curation_group" in df.columns and CURATION_GROUPS_TO_KEEP is not None:
        df = df[df["curation_group"].isin(CURATION_GROUPS_TO_KEEP)].copy()

    if MODEL_SHORT_TO_KEEP is not None and "model_short" in df.columns:
        df = df[df["model_short"].eq(MODEL_SHORT_TO_KEEP)].copy()

    if DROP_DUPLICATE_IMAGES:
        sort_cols = []
        if "curation_group" in df.columns:
            group_priority = {
                "fn_mel_all": 0,
                "fp_mel_top10": 1,
                "nearest_mel_centroid": 2,
                "nearest_nv_centroid": 3,
            }
            df["_group_priority"] = df["curation_group"].map(group_priority).fillna(99)
            sort_cols.append("_group_priority")

        if "image_id" in df.columns:
            sort_cols.append("image_id")

        if sort_cols:
            df = df.sort_values(sort_cols).copy()

        df = df.drop_duplicates("image_id", keep="first").reset_index(drop=True)
        df = df.drop(columns=[c for c in ["_group_priority"] if c in df.columns])

    if MAX_IMAGES is not None:
        df = df.head(MAX_IMAGES).copy()

    csv_dir = OUT_ROOT / "csv"
    csv_dir.mkdir(parents=True, exist_ok=True)

    active_csv = csv_dir / "bidirectional_cam_review_active.csv"
    df.to_csv(active_csv, index=False)

    print("ACTIVE_CSV:", active_csv)
    print("N images:", len(df))

    if "curation_group" in df.columns:
        display(df["curation_group"].value_counts())

    display(df[["image_id", "gt_label", "curation_group"]].head(20))

    return active_csv, df


ACTIVE_CSV, DISPLAY_DF = make_active_csv()
NUM_SAMPLES = len(DISPLAY_DF)

ACTIVE_CSV: /storage/homefs/cn21m021/projects/master-thesis/outputs/mel_nv/bidirectional_cam_review/csv/bidirectional_cam_review_active.csv
N images: 44


curation_group
nearest_nv_centroid     16
fp_mel_top10            13
nearest_mel_centroid    11
fn_mel_all               4
Name: count, dtype: int64

,image_id,gt_label,curation_group
0,ISIC_0024886,MEL,fn_mel_all
1,ISIC_0028173,MEL,fn_mel_all
2,ISIC_0029013,MEL,fn_mel_all
3,ISIC_0030552,MEL,fn_mel_all
4,ISIC_0024344,NV,fp_mel_top10
5,ISIC_0024944,NV,fp_mel_top10
6,ISIC_0025836,NV,fp_mel_top10
7,ISIC_0026209,NV,fp_mel_top10
8,ISIC_0026491,NV,fp_mel_top10
9,ISIC_0026584,NV,fp_mel_top10


In [3]:
def run_command(cmd: list[str], dry_run: bool = False):
    print("\n" + "=" * 100)
    print(" ".join(shlex.quote(str(x)) for x in cmd))
    print("=" * 100)
    if dry_run:
        return
    subprocess.run(cmd, cwd=REPO_ROOT, check=True)


def block_dir_name(block_index: int) -> str:
    return f"block_{block_index}".replace("-", "minus")


def generate_bidirectional_cams():
    panel_root = OUT_ROOT / "panels"
    panel_root.mkdir(parents=True, exist_ok=True)

    for scenario in SCENARIOS:
        for block_index in TARGET_BLOCK_INDICES:
            for direction in DIRECTION_CONFIGS:
                out_dir = (
                    panel_root
                    / scenario["short_name"]
                    / block_dir_name(block_index)
                    / direction["direction_name"]
                )
                out_dir.mkdir(parents=True, exist_ok=True)

                cmd = [
                    "python", "-m", "scripts.generate_finer_cam_panderm",
                    "--csv", str(ACTIVE_CSV),
                    "--image_col", "image_rel_path",
                    "--img_dir", str(IMG_DIR),
                    "--gt_col", GT_COL,
                    "--checkpoint", str(scenario["checkpoint"]),
                    "--checkpoint_model_type", scenario.get("checkpoint_model_type", "panderm"),
                    "--pooling", scenario["pooling"],
                    "--out_dir", str(out_dir),
                    "--num_samples", str(NUM_SAMPLES),
                    "--method", "finercam",
                    "--alpha", str(ALPHA),
                    "--panel_items", PANEL_ITEMS,
                    "--mask_root", str(MASK_ROOT),
                    "--mask_col", "mask_rel_path",
                    "--target_block_index", str(block_index),
                    "--clinician_labels",
                    "--model_display_name", f"{scenario['name']} block {block_index} {direction['display_name']}",
                    "--compare_mode", direction["compare_mode"],
                    "--A", "MEL",
                    "--B", "NV",
                    "--topk_compare", "1",
                    "--save_raw_cams",
                ]

                cmd += CLASS_ARGS

                print(
                    f"\nGenerating: {scenario['name']} | block {block_index} | {direction['display_name']}"
                )
                run_command(cmd, dry_run=DRY_RUN)


if RUN_CAM_GENERATION:
    generate_bidirectional_cams()
else:
    print("RUN_CAM_GENERATION=False, skipping CAM generation.")


Generating: CLS HA 0.5 | block -1 | GT direction

python -m scripts.generate_finer_cam_panderm --csv /storage/homefs/cn21m021/projects/master-thesis/outputs/mel_nv/bidirectional_cam_review/csv/bidirectional_cam_review_active.csv --image_col image_rel_path --img_dir /storage/homefs/cn21m021/projects/master-thesis/data/HAM10000 --gt_col gt_label --checkpoint /storage/homefs/cn21m021/projects/master-thesis/external/checkpoints4/checkpoint-best-cls.pth --checkpoint_model_type panderm --pooling cls --out_dir /storage/homefs/cn21m021/projects/master-thesis/outputs/mel_nv/bidirectional_cam_review/panels/cls_ha05/block_minus1/gt_direction --num_samples 44 --method finercam --alpha 0.8 --panel_items rgb_gt_mask,gradcam_a,map_diff,finercam --mask_root /storage/homefs/cn21m021/projects/master-thesis/data/HAM10000 --mask_col mask_rel_path --target_block_index -1 --clinician_labels --model_display_name 'CLS HA 0.5 block -1 GT direction' --compare_mode gt_pair --A MEL --B NV --topk_compare 1 --save

/storage/homefs/cn21m021/projects/master-thesis/scripts/generate_finer_cam_panderm.py:716: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  ckpt = torch.load(checkpoint_path, m

[warn] load_state_dict mismatch: missing=2, unexpected=2
  missing sample: ['norm.weight', 'norm.bias']
  unexpected sample: ['fc_norm.weight', 'fc_norm.bias']
[info] Loaded PanDerm Base FT from checkpoint-best-cls.pth
[info] checkpoint_model_type= panderm pooling= cls use_seg_gate= False seg_gate_bg_keep= 0.15 seg_gate_detach= True
normalization method:  imagenet
[info] CAM target layer: blocks[11].norm1 (requested -1)
[debug compare_mode] gt_pair | A_idx=0 (MEL) | B_idx=1 (NV) | comparison=[1]
[saved raw cams] /storage/homefs/cn21m021/projects/master-thesis/outputs/mel_nv/bidirectional_cam_review/panels/cls_ha05/block_minus1/gt_direction/raw_cams/ISIC_0024886
[info] images/ISIC_0024886.jpg: A=0(MEL)  B=1(NV)  comparison=[NV]  top3=[NV: 0.862, MEL: 0.138]
[debug compare_mode] gt_pair | A_idx=0 (MEL) | B_idx=1 (NV) | comparison=[1]
[saved raw cams] /storage/homefs/cn21m021/projects/master-thesis/outputs/mel_nv/bidirectional_cam_review/panels/cls_ha05/block_minus1/gt_direction/raw_cams/

/storage/homefs/cn21m021/projects/master-thesis/scripts/generate_finer_cam_panderm.py:716: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  ckpt = torch.load(checkpoint_path, m

[warn] load_state_dict mismatch: missing=2, unexpected=2
  missing sample: ['norm.weight', 'norm.bias']
  unexpected sample: ['fc_norm.weight', 'fc_norm.bias']
[info] Loaded PanDerm Base FT from checkpoint-best-cls.pth
[info] checkpoint_model_type= panderm pooling= cls use_seg_gate= False seg_gate_bg_keep= 0.15 seg_gate_detach= True
normalization method:  imagenet
[info] CAM target layer: blocks[11].norm1 (requested -1)
[debug compare_mode] inverse_gt_pair | A_idx=1 (NV) | B_idx=0 (MEL) | comparison=[0]
[saved raw cams] /storage/homefs/cn21m021/projects/master-thesis/outputs/mel_nv/bidirectional_cam_review/panels/cls_ha05/block_minus1/inverse_direction/raw_cams/ISIC_0024886
[info] images/ISIC_0024886.jpg: A=1(NV)  B=0(MEL)  comparison=[MEL]  top3=[NV: 0.862, MEL: 0.138]
[debug compare_mode] inverse_gt_pair | A_idx=1 (NV) | B_idx=0 (MEL) | comparison=[0]
[saved raw cams] /storage/homefs/cn21m021/projects/master-thesis/outputs/mel_nv/bidirectional_cam_review/panels/cls_ha05/block_minus1/

...

[info] Loaded PanDerm Base FT from checkpoint-best-gap.pth
[info] checkpoint_model_type= panderm pooling= mean use_seg_gate= False seg_gate_bg_keep= 0.15 seg_gate_detach= True
normalization method:  imagenet
[info] CAM target layer: blocks[8].norm1 (requested -4)
[debug compare_mode] inverse_gt_pair | A_idx=1 (NV) | B_idx=0 (MEL) | comparison=[0]
[saved raw cams] /storage/homefs/cn21m021/projects/master-thesis/outputs/mel_nv/bidirectional_cam_review/panels/gap_ha025/block_minus4/inverse_direction/raw_cams/ISIC_0024886
[info] images/ISIC_0024886.jpg: A=1(NV)  B=0(MEL)  comparison=[MEL]  top3=[NV: 0.662, MEL: 0.338]
[debug compare_mode] inverse_gt_pair | A_idx=1 (NV) | B_idx=0 (MEL) | comparison=[0]
[saved raw cams] /storage/homefs/cn21m021/projects/master-thesis/outputs/mel_nv/bidirectional_cam_review/panels/gap_ha025/block_minus4/inverse_direction/raw_cams/ISIC_0028173
[info] images/ISIC_0028173.jpg: A=1(NV)  B=0(MEL)  comparison=[MEL]  top3=[NV: 0.696, MEL: 0.304]
[debug compare_mode]

/storage/homefs/cn21m021/projects/master-thesis/scripts/generate_finer_cam_panderm.py:716: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  ckpt = torch.load(checkpoint_path, m

[info] Loaded PanDerm Base FT from checkpoint-best-gap.pth
[info] checkpoint_model_type= panderm pooling= mean use_seg_gate= False seg_gate_bg_keep= 0.15 seg_gate_detach= True
normalization method:  imagenet
[info] CAM target layer: blocks[2].norm1 (requested -10)
[debug compare_mode] gt_pair | A_idx=0 (MEL) | B_idx=1 (NV) | comparison=[1]
[saved raw cams] /storage/homefs/cn21m021/projects/master-thesis/outputs/mel_nv/bidirectional_cam_review/panels/gap_ha025/block_minus10/gt_direction/raw_cams/ISIC_0024886
[info] images/ISIC_0024886.jpg: A=0(MEL)  B=1(NV)  comparison=[NV]  top3=[NV: 0.662, MEL: 0.338]
[debug compare_mode] gt_pair | A_idx=0 (MEL) | B_idx=1 (NV) | comparison=[1]
[saved raw cams] /storage/homefs/cn21m021/projects/master-thesis/outputs/mel_nv/bidirectional_cam_review/panels/gap_ha025/block_minus10/gt_direction/raw_cams/ISIC_0028173
[info] images/ISIC_0028173.jpg: A=0(MEL)  B=1(NV)  comparison=[NV]  top3=[NV: 0.696, MEL: 0.304]
[debug compare_mode] gt_pair | A_idx=0 (MEL) 

/storage/homefs/cn21m021/projects/master-thesis/scripts/generate_finer_cam_panderm.py:716: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  ckpt = torch.load(checkpoint_path, m

[info] Loaded PanDerm Base FT from checkpoint-best-gap.pth
[info] checkpoint_model_type= panderm pooling= mean use_seg_gate= False seg_gate_bg_keep= 0.15 seg_gate_detach= True
normalization method:  imagenet
[info] CAM target layer: blocks[2].norm1 (requested -10)
[debug compare_mode] inverse_gt_pair | A_idx=1 (NV) | B_idx=0 (MEL) | comparison=[0]
[saved raw cams] /storage/homefs/cn21m021/projects/master-thesis/outputs/mel_nv/bidirectional_cam_review/panels/gap_ha025/block_minus10/inverse_direction/raw_cams/ISIC_0024886
[info] images/ISIC_0024886.jpg: A=1(NV)  B=0(MEL)  comparison=[MEL]  top3=[NV: 0.662, MEL: 0.338]
[debug compare_mode] inverse_gt_pair | A_idx=1 (NV) | B_idx=0 (MEL) | comparison=[0]
[saved raw cams] /storage/homefs/cn21m021/projects/master-thesis/outputs/mel_nv/bidirectional_cam_review/panels/gap_ha025/block_minus10/inverse_direction/raw_cams/ISIC_0028173
[info] images/ISIC_0028173.jpg: A=1(NV)  B=0(MEL)  comparison=[MEL]  top3=[NV: 0.696, MEL: 0.304]
[debug compare_mo

In [4]:
def get_font(size: int, bold: bool = False):
    candidates = [
        "/System/Library/Fonts/Supplemental/Arial Bold.ttf" if bold else "/System/Library/Fonts/Supplemental/Arial.ttf",
        "/usr/share/fonts/truetype/dejavu/DejaVuSans-Bold.ttf" if bold else "/usr/share/fonts/truetype/dejavu/DejaVuSans.ttf",
    ]
    for path in candidates:
        if path and Path(path).exists():
            return ImageFont.truetype(path, size=size)
    return ImageFont.load_default()


FONT_TITLE = get_font(34, bold=True)
FONT_SUBTITLE = get_font(22, bold=False)
FONT_LABEL = get_font(22, bold=True)
FONT_SMALL = get_font(17, bold=False)


def image_id_to_stem(image_id_value: str) -> str:
    p = Path(str(image_id_value))
    stem = p.stem if p.suffix else p.name
    return stem.replace("/", "_").replace("\\", "_").replace(" ", "_")


def find_panel_png(scenario: dict, block_index: int, direction_name: str, row: pd.Series) -> Path | None:
    out_dir = (
        OUT_ROOT
        / "panels"
        / scenario["short_name"]
        / block_dir_name(block_index)
        / direction_name
    )

    candidates = []
    if "image_rel_path" in row and pd.notna(row["image_rel_path"]):
        candidates.append(image_id_to_stem(row["image_rel_path"]))
    if "image_id" in row and pd.notna(row["image_id"]):
        candidates.append(image_id_to_stem(row["image_id"]))

    for stem in dict.fromkeys(candidates):
        direct = out_dir / f"{stem}_{PANEL_SUFFIX}.png"
        if direct.exists():
            return direct

        matches = sorted(out_dir.glob(f"{stem}_*.png"))
        if matches:
            return matches[0]

    return None


def load_and_resize_panel(panel_path: Path, target_width: int) -> Image.Image:
    panel = Image.open(panel_path).convert("RGB")
    scale = target_width / panel.width
    new_h = int(panel.height * scale)
    return panel.resize((target_width, new_h), Image.Resampling.LANCZOS)

In [5]:
def make_page_for_image_and_model(row: pd.Series, scenario: dict) -> Image.Image:
    page_width = 2300
    margin = 45
    label_width = 250
    gap = 14
    title_h = 145

    available_panel_width = page_width - 2 * margin - label_width - gap

    loaded_rows = []

    for block_index in TARGET_BLOCK_INDICES:
        for direction in DIRECTION_CONFIGS:
            panel_path = find_panel_png(
                scenario=scenario,
                block_index=block_index,
                direction_name=direction["direction_name"],
                row=row,
            )

            if panel_path is None:
                panel = None
            else:
                panel = load_and_resize_panel(panel_path, available_panel_width)

            loaded_rows.append({
                "block": block_index,
                "direction_name": direction["direction_name"],
                "direction_display": direction["display_name"],
                "panel": panel,
                "panel_path": panel_path,
            })

    row_heights = [r["panel"].height if r["panel"] is not None else 190 for r in loaded_rows]
    page_height = title_h + margin + sum(row_heights) + gap * (len(row_heights) - 1) + margin

    page = Image.new("RGB", (page_width, page_height), "white")
    draw = ImageDraw.Draw(page)

    image_id = row.get("image_id", row.get("image_rel_path", "unknown"))
    gt = row.get(GT_COL, row.get("gt_label", "unknown"))
    group = row.get("curation_group", "unknown_group")
    pred = row.get("pred_label", "unknown_pred")
    correctness = row.get("correctness", "unknown_correctness")
    mel_prob = row.get("mel_probability", None)

    title = f"Bidirectional CAM review: {scenario['name']}"
    subtitle = f"Image: {image_id} | GT: {gt} | group: {group} | pred: {pred} | {correctness}"

    if mel_prob is not None and pd.notna(mel_prob):
        subtitle += f" | MEL prob: {float(mel_prob):.3f}"

    draw.text((margin, 26), title, fill="black", font=FONT_TITLE)
    draw.text((margin, 78), subtitle, fill=(60, 60, 60), font=FONT_SUBTITLE)

    explanation = "Row meaning: GT direction = evidence for true class vs other class. Inverse direction = evidence for other class vs true class."
    draw.text((margin, 112), explanation, fill=(90, 90, 90), font=FONT_SMALL)

    y = title_h

    for r in loaded_rows:
        row_h = r["panel"].height if r["panel"] is not None else 190

        label_x = margin
        label_y = y + 18

        draw.text((label_x, label_y), f"Block {r['block']}", fill="black", font=FONT_LABEL)

        if r["direction_name"] == "gt_direction":
            direction_text = "GT vs other"
        else:
            direction_text = "Other vs GT"

        draw.text((label_x, label_y + 34), direction_text, fill=(70, 70, 70), font=FONT_SMALL)

        if r["panel"] is None:
            box_x = margin + label_width + gap
            box_y = y
            draw.rectangle(
                [box_x, box_y, box_x + available_panel_width, box_y + row_h],
                outline=(180, 180, 180),
                width=2,
            )
            draw.text(
                (box_x + 30, box_y + 65),
                "Missing panel PNG",
                fill=(160, 0, 0),
                font=FONT_LABEL,
            )
        else:
            page.paste(r["panel"], (margin + label_width + gap, y))

        y += row_h + gap

    return page


def build_pdf_for_scenario(scenario: dict):
    pages = []

    for _, row in DISPLAY_DF.iterrows():
        pages.append(make_page_for_image_and_model(row, scenario))

    if not pages:
        raise RuntimeError("No pages generated.")

    pdf_out = OUT_ROOT / f"bidirectional_cam_review_{scenario['short_name']}.pdf"
    pages[0].save(pdf_out, save_all=True, append_images=pages[1:], resolution=150.0)

    print("Saved PDF:", pdf_out)
    return pdf_out


PDF_OUTPUTS = []

if DRY_RUN:
    print("DRY_RUN=True, skipping PDF build.")
elif not BUILD_PDFS:
    print("BUILD_PDFS=False, skipping PDF build.")
else:
    for scenario in SCENARIOS:
        PDF_OUTPUTS.append(build_pdf_for_scenario(scenario))

PDF_OUTPUTS

Saved PDF: /storage/homefs/cn21m021/projects/master-thesis/outputs/mel_nv/bidirectional_cam_review/bidirectional_cam_review_cls_ha05.pdf
Saved PDF: /storage/homefs/cn21m021/projects/master-thesis/outputs/mel_nv/bidirectional_cam_review/bidirectional_cam_review_gap_ha025.pdf


[PosixPath('/storage/homefs/cn21m021/projects/master-thesis/outputs/mel_nv/bidirectional_cam_review/bidirectional_cam_review_cls_ha05.pdf'),
 PosixPath('/storage/homefs/cn21m021/projects/master-thesis/outputs/mel_nv/bidirectional_cam_review/bidirectional_cam_review_gap_ha025.pdf')]

In [6]:
def make_page_for_image_all_models(row: pd.Series) -> Image.Image:
    model_pages = [make_page_for_image_and_model(row, scenario) for scenario in SCENARIOS]

    width = max(p.width for p in model_pages)
    gap = 35
    height = sum(p.height for p in model_pages) + gap * (len(model_pages) - 1)

    combined = Image.new("RGB", (width, height), "white")

    y = 0
    for p in model_pages:
        combined.paste(p, (0, y))
        y += p.height + gap

    return combined


def build_combined_pdf():
    pages = []

    for _, row in DISPLAY_DF.iterrows():
        pages.append(make_page_for_image_all_models(row))

    if not pages:
        raise RuntimeError("No pages generated.")

    pdf_out = OUT_ROOT / "bidirectional_cam_review_combined_cls_gap.pdf"
    pages[0].save(pdf_out, save_all=True, append_images=pages[1:], resolution=150.0)

    print("Saved combined PDF:", pdf_out)
    return pdf_out


COMBINED_PDF = build_combined_pdf() if BUILD_PDFS and not DRY_RUN else None
COMBINED_PDF

Saved combined PDF: /storage/homefs/cn21m021/projects/master-thesis/outputs/mel_nv/bidirectional_cam_review/bidirectional_cam_review_combined_cls_gap.pdf


PosixPath('/storage/homefs/cn21m021/projects/master-thesis/outputs/mel_nv/bidirectional_cam_review/bidirectional_cam_review_combined_cls_gap.pdf')

In [7]:
config = {
    "active_csv": str(ACTIVE_CSV),
    "curation_csv": str(CURATION_CSV),
    "out_root": str(OUT_ROOT),
    "num_samples": NUM_SAMPLES,
    "target_block_indices": TARGET_BLOCK_INDICES,
    "panel_items": PANEL_ITEMS,
    "alpha": ALPHA,
    "directions": DIRECTION_CONFIGS,
    "scenarios": [
        {
            "name": s["name"],
            "short_name": s["short_name"],
            "checkpoint": str(s["checkpoint"]),
            "checkpoint_model_type": s.get("checkpoint_model_type", "panderm"),
            "pooling": s["pooling"],
        }
        for s in SCENARIOS
    ],
    "pdf_outputs": [str(p) for p in PDF_OUTPUTS],
    "combined_pdf": str(COMBINED_PDF) if COMBINED_PDF is not None else None,
}

config_out = OUT_ROOT / "bidirectional_cam_review_config.json"
config_out.write_text(json.dumps(config, indent=2))

print("Saved config:", config_out)

for scenario in SCENARIOS:
    print("\n" + "=" * 80)
    print(scenario["name"])

    for block_index in TARGET_BLOCK_INDICES:
        for direction in DIRECTION_CONFIGS:
            out_dir = (
                OUT_ROOT
                / "panels"
                / scenario["short_name"]
                / block_dir_name(block_index)
                / direction["direction_name"]
            )
            pngs = sorted(out_dir.glob("*.png"))
            raw_dir = out_dir / "raw_cams"
            raw_files = sorted(raw_dir.glob("*.npy")) if raw_dir.exists() else []

            print(
                f"  block {block_index:>3} | {direction['direction_name']:<18}: "
                f"png={len(pngs):>4} raw={len(raw_files):>4}"
            )

print("\nPDF outputs:")
for p in PDF_OUTPUTS:
    print(" ", p)

print("\nCombined PDF:")
print(" ", COMBINED_PDF)

Saved config: /storage/homefs/cn21m021/projects/master-thesis/outputs/mel_nv/bidirectional_cam_review/bidirectional_cam_review_config.json

CLS HA 0.5
  block  -1 | gt_direction      : png=  44 raw= 176
  block  -1 | inverse_direction : png=  44 raw= 176
  block  -4 | gt_direction      : png=  44 raw= 176
  block  -4 | inverse_direction : png=  44 raw= 176
  block -10 | gt_direction      : png=  44 raw= 176
  block -10 | inverse_direction : png=  44 raw= 176

GAP HA 0.25
  block  -1 | gt_direction      : png=  44 raw= 176
  block  -1 | inverse_direction : png=  44 raw= 176
  block  -4 | gt_direction      : png=  44 raw= 176
  block  -4 | inverse_direction : png=  44 raw= 176
  block -10 | gt_direction      : png=  44 raw= 176
  block -10 | inverse_direction : png=  44 raw= 176

PDF outputs:
  /storage/homefs/cn21m021/projects/master-thesis/outputs/mel_nv/bidirectional_cam_review/bidirectional_cam_review_cls_ha05.pdf
  /storage/homefs/cn21m021/projects/master-thesis/outputs/mel_nv/bidir